# Génération de corpus synthétique avec DSPy

Ce carnet sert à générer un corpus d'entraînement **instructionnel** pour une tâche d'extraction d'expressions caractéristiques.

Format cible par ligne dans le JSONL :

```json
{
  "instruction": "Extraire les expressions caractéristiques du texte suivant.",
  "input": "<texte>",
  "output": ["expression 1", "expression 2", "expression 3"],
  "metadata": {
    "theme": "...",
    "categorie": "...",
    "type_du_document": "...",
    "url": "...",
    "date": "..."
  }
}
```

L'objectif est de préparer un corpus utilisable ensuite pour un fine-tuning LoRA d'un petit modèle de type Qwen.

## 1. Dépendances

Ce carnet suppose :

- `dspy`
- un endpoint compatible OpenAI **ou** une API OpenAI
- quatre fichiers de contrôle :
  - `prompt.txt`
  - `thèmes.json`
  - `catégories.jsonl`
  - `types_de_document_par_catégorie.json`

In [3]:
# Si nécessaire :
# !pip install -U dspy-ai

## 2. Configuration

In [10]:
from __future__ import annotations

import ast
import json
import os
import random
import re
import time
from dataclasses import dataclass
from typing import Dict, List, Optional, Any

import dspy
from dspy.clients import LM, configure_cache as dspy_configure_cache

os.environ.setdefault("DSPY_CACHEDIR", os.path.join(os.getcwd(), ".dspy_cache"))
os.makedirs(os.environ["DSPY_CACHEDIR"], exist_ok=True)

try:
    dspy_configure_cache(enable_disk_cache=False, enable_memory_cache=True)
except Exception:
    pass

dspy.settings.cache = None

@dataclass
class Config:
    prompt_path: str = "./prompt.txt"
    themes_path: str = "./thèmes.json"
    categories_path: str = "./catégories.jsonl"
    types_map_path: str = "./types_de_document_par_catégorie.json"
    output_path: str = "./corpus_instruction_qwen.jsonl"

    provider: str = "openai"
    model: str = "deep-chat"
    temperature: float = 0.7

    count: int = 100
    max_attempts_per_item: int = 3

CFG = Config()
CFG

Config(prompt_path='./prompt.txt', themes_path='./thèmes.json', categories_path='./catégories.jsonl', types_map_path='./types_de_document_par_catégorie.json', output_path='./corpus_instruction_qwen.jsonl', provider='openai', model='deep-chat', temperature=0.7, count=100, max_attempts_per_item=3)

## 3. Chargement des fichiers auxiliaires

In [12]:
def load_prompt(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read().strip()

def _extract_python_rhs_container(text: str, open_ch: str, close_ch: str) -> str:
    start = text.find(open_ch)
    if start == -1:
        raise ValueError("Début de conteneur non trouvé")
    depth = 0
    for i in range(start, len(text)):
        if text[i] == open_ch:
            depth += 1
        elif text[i] == close_ch:
            depth -= 1
            if depth == 0:
                return text[start:i+1]
    raise ValueError("Fin de conteneur non trouvée")

def load_themes(path: str) -> List[str]:
    with open(path, "r", encoding="utf-8") as f:
        data = f.read()
    list_src = _extract_python_rhs_container(data, "[", "]")
    themes = ast.literal_eval(list_src)
    if not isinstance(themes, list) or not all(isinstance(x, str) for x in themes):
        raise ValueError("Format des thèmes invalide")
    return themes

def load_categories(path: str) -> List[str]:
    with open(path, "r", encoding="utf-8") as f:
        raw = f.read()

    try:
        parsed = json.loads(raw)
        if isinstance(parsed, list) and all(isinstance(x, str) for x in parsed):
            return parsed
    except Exception:
        pass

    body = raw.strip().lstrip().lstrip("[").rstrip().rstrip("]")
    items = []
    for line in body.splitlines():
        token = line.strip().rstrip(",")
        if not token:
            continue
        if (token.startswith('"') and token.endswith('"')) or (token.startswith("'") and token.endswith("'")):
            token = token[1:-1]
        items.append(token)

    items = [x.strip() for x in items if x.strip()]
    items = [re.sub(r"[^a-zA-Z0-9_\-àâäéèêëîïôöùûüç]+$", "", x) for x in items]
    return items

def load_types_by_category(path: str) -> Dict[str, List[str]]:
    with open(path, "r", encoding="utf-8") as f:
        data = f.read()
    dict_src = _extract_python_rhs_container(data, "{", "}")
    mapping = ast.literal_eval(dict_src)
    if not isinstance(mapping, dict) or not all(isinstance(v, list) for v in mapping.values()):
        raise ValueError("Format du mapping invalide")
    return mapping

prompt_text = load_prompt(CFG.prompt_path)
themes = load_themes(CFG.themes_path)
categories = load_categories(CFG.categories_path)
types_map = load_types_by_category(CFG.types_map_path)

len(themes), len(categories), list(types_map)[:5]

(37, 10, ['extrait', 'commentaire', 'synthese', 'resume', 'note_de_lecture'])

## 4. Configuration du modèle DSPy

In [14]:
# DeepSeek
lm = dspy.LM(
    "openai/deepseek-chat",          # modèle DeepSeek côté API
    api_base="https://api.deepseek.com",  # ou https://api.deepseek.com/v1
    api_key=os.environ["DEEPSEEK_API_KEY"],
    model_type="chat",
)
dspy.settings.cache = None
dspy.settings.configure(lm=lm)
lm("Dites bonjour, répondez en un mot")

['Bonjour']

## 5. Signature DSPy

In [15]:
class CorpusSignature(dspy.Signature):
    """Produire strictement les champs demandés pour une note synthétique.
    Aucun commentaire hors contenu des champs.
    Les expressions_clefs doivent être présentes textuellement dans le contenu.
    """

    theme: str = dspy.InputField(desc="Thème principal de la note")
    categorie: str = dspy.InputField(desc="Type de note : extrait, commentaire, synthese, texte_libre")

    contenu: str = dspy.OutputField(desc="Contenu de la note")

def build_predictor(instruction: str) -> dspy.Predict:
    CorpusSignature.__doc__ = (CorpusSignature.__doc__ or "") + "\n\n" + instruction
    return dspy.Predict(CorpusSignature)

predictor = build_predictor(prompt_text)

## 6. Fonctions de génération et de validation légère

In [16]:
def choose_triplet(themes: List[str], categories: List[str], types_map: Dict[str, List[str]]):
    theme = random.choice(themes)
    categorie = random.choice(categories)
    available_types = types_map.get(categorie, ["document"])
    type_du_document = random.choice(available_types)
    return theme, categorie, type_du_document

def normalize_expressions(value: Any) -> List[str]:
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]
    if isinstance(value, str):
        value = value.strip()
        try:
            parsed = json.loads(value)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except Exception:
            pass
        return [x.strip() for x in value.split(",") if x.strip()]
    return []

def validate_sample(sample: Dict[str, Any]) -> bool:
    contenu = sample["contenu"].strip()
    expressions = normalize_expressions(sample["expressions_clefs"])

    if len(contenu) < 80:
        return False
    if len(expressions) < 3:
        return False
    if len(set(expressions)) != len(expressions):
        return False
    if not all(expr in contenu for expr in expressions):
        return False
    return True

def generate_one(predictor: dspy.Predict, theme: str, categorie: str, type_du_document: str) -> Optional[Dict[str, Any]]:
    pred = predictor(theme=theme, categorie=categorie, type_du_document=type_du_document)

    sample = {
        "contenu": str(pred.contenu).strip(),
        "expressions_clefs": normalize_expressions(pred.expressions_clefs),
        "url": str(pred.url).strip(),
        "date": str(pred.date).strip(),
        "categorie": categorie,
        "theme": theme,
        "type_du_document": type_du_document,
    }
    return sample if validate_sample(sample) else None

def to_instruction_record(sample: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "instruction": "Extraire les expressions caractéristiques du texte suivant.",
        "input": sample["contenu"],
        "output": sample["expressions_clefs"],
        "metadata": {
            "theme": sample["theme"],
            "categorie": sample["categorie"],
            "type_du_document": sample["type_du_document"],
            "url": sample["url"],
            "date": sample["date"],
        },
    }

## 7. Test sur un exemple

In [17]:
theme, categorie, type_du_document = choose_triplet(themes, categories, types_map)
example = generate_one(predictor, theme, categorie, type_du_document)
example

{'contenu': "L'ouvrage propose une analyse systémique de la crise écologique, dépassant la simple approche technicienne. L'auteur défend l'idée que la transition requiert une transformation profonde des imaginaires économiques et sociaux, et non seulement des innovations vertes. Il insiste sur le concept de « limites planétaires » comme cadre incontournable pour repenser la prospérité. La notion d'« anthropocène » y est critiquée pour son caractère trop général, lui préférant celle d'« capitalocène » qui pointe les logiques d'accumulation comme cause structurelle. La conclusion appelle à une réorientation vers des économies de la sobriété et de la régénération des écosystèmes.",
 'expressions_clefs': ['limites planétaires',
  'anthropocène',
  'capitalocène',
  'économies de la sobriété',
  'régénération des écosystèmes'],
 'url': 'ISBN 978-2-1234-5678-9',
 'date': '2023-05-17',
 'categorie': 'resume',
 'theme': 'écologie',
 'type_du_document': 'ouvrage académique'}

## 8. Génération du corpus complet

In [23]:
def build_corpus(count: int, predictor: dspy.Predict) -> List[Dict[str, Any]]:
    records = []
    attempts_global = 0
    max_global_attempts = count * CFG.max_attempts_per_item * 4
    started = time.time()

    while len(records) < count and attempts_global < max_global_attempts:
        theme, categorie, type_du_document = choose_triplet(themes, categories, types_map)
        print(f"{len(records)}/{count}")
        for _ in range(CFG.max_attempts_per_item):
            attempts_global += 1
            try:
                sample = generate_one(predictor, theme, categorie, type_du_document)
                print(f"thème = {theme}, catégorie = {categorie}, type de document = {type_du_document}")
            except Exception as e:
                print(e)
                sample = None

            if sample is None:
                continue

            records.append(to_instruction_record(sample))
            break

        if len(records) % 10 == 0 and len(records) > 0:
            dt = time.time() - started
            print(f"{len(records)} échantillons valides en {dt:.1f}s")

    return records

records = build_corpus(CFG.count, predictor)
len(records)

0/100
thème = économie politique, catégorie = question, type de document = séminaire
1/100
thème = mathématiques, catégorie = note_de_lecture, type de document = ouvrage d'histoire
2/100
thème = histoire de l'Europe, catégorie = idee, type de document = réflexion personnelle
thème = histoire de l'Europe, catégorie = idee, type de document = réflexion personnelle
thème = histoire de l'Europe, catégorie = idee, type de document = réflexion personnelle
2/100
thème = open source, catégorie = note_de_lecture, type de document = roman
3/100
thème = histoire de la science, catégorie = observation, type de document = observation de terrain
thème = histoire de la science, catégorie = observation, type de document = observation de terrain
thème = histoire de la science, catégorie = observation, type de document = observation de terrain
3/100
thème = roman du XIXe siècle, catégorie = resume, type de document = article Wikipédia
4/100
thème = sociologie, catégorie = commentaire, type de document =

100

## 9. Sauvegarde en JSONL

In [24]:
with open(CFG.output_path, "w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Corpus écrit dans : {CFG.output_path}")

Corpus écrit dans : ./corpus_instruction_qwen.jsonl


## 10. Inspection rapide

In [25]:
records[:2]

[{'instruction': 'Extraire les expressions caractéristiques du texte suivant.',
  'input': "L'intervenant a soulevé une question centrale : dans quelle mesure la politique monétaire expansionniste, poursuivie depuis la crise de 2008, a-t-elle réellement atteint son objectif de stimulation de l'investissement productif ? Les données présentées suggèrent un découplage croissant entre la liquidité injectée dans le système financier et les investissements en capital fixe des entreprises. Cela pose le problème de l'efficacité des canaux de transmission traditionnels. La discussion qui a suivi a porté sur les alternatives possibles, comme un ciblage plus direct du crédit ou des politiques fiscales coordonnées. La question reste ouverte et mérite un examen approfondi des mécanismes de redistribution de la liquidité.",
  'output': ['politique monétaire expansionniste',
   'investissement productif',
   'canaux de transmission',
   'liquidité injectée',
   'politiques fiscales coordonnées'],
  

## 11. Variante conversationnelle

Plus tard, vous pourrez convertir ce format vers un dataset `messages` pour SFT conversationnel.
Pour démarrer, le format `instruction / input / output` reste le plus simple.